In [45]:
# Cell 1: Import libraries
import pandas as pd
import geopandas as gpd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import joblib
import os

load_dotenv()
print("Libraries imported successfully!")

Libraries imported successfully!


In [46]:
# Cell 2: Load all shapefiles exported from QGIS
processed_dir = r'D:\sfpris\data\processed'

# Load HL incidents with fault distance
hl_incidents = gpd.read_file(f"{processed_dir}/hl_incidents_fault_distance.shp")
print(f"HL incidents: {hl_incidents.shape}")

# Load Gas incidents with fault distance
gas_incidents = gpd.read_file(f"{processed_dir}/gas_incidents_fault_distance.shp")
print(f"Gas incidents: {gas_incidents.shape}")

# Load earthquake data
earthquakes = gpd.read_file(f"{processed_dir}/usgs_earthquakes.shp")
print(f"Earthquakes: {earthquakes.shape}")

# Load HL pipelines
hl_pipes = gpd.read_file(f"{processed_dir}/hl_pipelines_earthquake_join.shp")
print(f"HL pipelines: {hl_pipes.shape}")

# Load Gas pipelines
gas_pipes = gpd.read_file(f"{processed_dir}/gas_pipelines_earthquake_join.shp")
print(f"Gas pipelines: {gas_pipes.shape}")

HL incidents: (5951, 26)
Gas incidents: (2069, 27)
Earthquakes: (219922, 8)
HL pipelines: (236, 11)
Gas pipelines: (32961, 12)


In [47]:
# Cell 3: Convert CRS to meters and create 50km buffers around pipelines
hl_pipes = hl_pipes.to_crs(epsg=3857)
gas_pipes = gas_pipes.to_crs(epsg=3857)
earthquakes = earthquakes.to_crs(epsg=3857)

# Create 50km buffer around pipelines
hl_pipes_buffer = hl_pipes.copy()
hl_pipes_buffer['geometry'] = hl_pipes.buffer(50000)

gas_pipes_buffer = gas_pipes.copy()
gas_pipes_buffer['geometry'] = gas_pipes.buffer(50000)

print("CRS converted and buffers created successfully!")
print(f"HL pipeline buffers: {hl_pipes_buffer.shape}")
print(f"Gas pipeline buffers: {gas_pipes_buffer.shape}")

CRS converted and buffers created successfully!
HL pipeline buffers: (236, 11)
Gas pipeline buffers: (32961, 12)


In [49]:
# Cell 4: Spatial join earthquakes to pipeline buffers
# HL pipelines
hl_eq_join = gpd.sjoin(hl_pipes_buffer, earthquakes, how='left', predicate='contains')
hl_eq_stats = hl_eq_join.groupby(hl_eq_join.index).agg(
    eq_count=('magnitude_right', 'count'),
    eq_max_mag=('magnitude_right', 'max'),
    eq_avg_depth=('depth_right', 'mean')
).reset_index()

# Fill NaN with 0
hl_eq_stats['eq_max_mag'] = hl_eq_stats['eq_max_mag'].fillna(0)
hl_eq_stats['eq_avg_depth'] = hl_eq_stats['eq_avg_depth'].fillna(0)

# Gas pipelines
gas_eq_join = gpd.sjoin(gas_pipes_buffer, earthquakes, how='left', predicate='contains')
gas_eq_stats = gas_eq_join.groupby(gas_eq_join.index).agg(
    eq_count=('magnitude_right', 'count'),
    eq_max_mag=('magnitude_right', 'max'),
    eq_avg_depth=('depth_right', 'mean')
).reset_index()

# Fill NaN with 0
gas_eq_stats['eq_max_mag'] = gas_eq_stats['eq_max_mag'].fillna(0)
gas_eq_stats['eq_avg_depth'] = gas_eq_stats['eq_avg_depth'].fillna(0)

print(f"HL earthquake stats: {hl_eq_stats.shape}")
print(f"Gas earthquake stats: {gas_eq_stats.shape}")
print(hl_eq_stats.head())

HL earthquake stats: (236, 4)
Gas earthquake stats: (32961, 4)
   index  eq_count  eq_max_mag  eq_avg_depth
0      0         0         0.0         0.000
1      1         0         0.0         0.000
2      2         2         3.2         8.895
3      3         7         3.3         0.000
4      4         2         3.2         3.000


In [50]:
# Cell 5: Merge earthquake stats back to pipelines
hl_pipes_final = hl_pipes.merge(hl_eq_stats, left_index=True, right_on='index')
gas_pipes_final = gas_pipes.merge(gas_eq_stats, left_index=True, right_on='index')

print(f"HL pipes final: {hl_pipes_final.shape}")
print(f"Gas pipes final: {gas_pipes_final.shape}")

HL pipes final: (236, 15)
Gas pipes final: (32961, 16)


In [51]:
# Cell 6: Convert incidents CRS and spatial join with pipeline earthquake stats
hl_incidents = hl_incidents.to_crs(epsg=3857)
gas_incidents = gas_incidents.to_crs(epsg=3857)

# Spatial join HL incidents with HL pipeline earthquake stats
hl_final = gpd.sjoin_nearest(
    hl_incidents,
    hl_pipes_final[['Pipename', 'eq_count', 'eq_max_mag', 'eq_avg_depth', 'geometry']],
    how='left'
)

# Spatial join Gas incidents with Gas pipeline earthquake stats
gas_final = gpd.sjoin_nearest(
    gas_incidents,
    gas_pipes_final[['TYPEPIPE', 'eq_count', 'eq_max_mag', 'eq_avg_depth', 'geometry']],
    how='left'
)

# Remove duplicates
hl_final = hl_final.drop_duplicates(subset='REPORT_NUM', keep='first')
gas_final = gas_final.drop_duplicates(subset='REPORT_NUM', keep='first')

print(f"HL final: {hl_final.shape}")
print(f"Gas final: {gas_final.shape}")
print(f"\nHL final columns: {hl_final.columns.tolist()}")

HL final: (5940, 31)
Gas final: (2018, 32)

HL final columns: ['REPORT_NUM', 'IYEAR', 'SIGNIFICAN', 'CAUSE', 'LOCATION_L', 'LOCATION_1', 'ONSHORE_ST', 'NAME', 'TOTAL_COST', 'PIPE_DIAME', 'MATERIAL_I', 'INSTALLATI', 'SYSTEM_PAR', 'source', 'Opername', 'Pipename_left', 'Shape_Leng', 'n', 'distance', 'feature_x', 'feature_y', 'nearest_x', 'nearest_y', 'HubName', 'HubDist', 'geometry', 'index_right', 'Pipename_right', 'eq_count', 'eq_max_mag', 'eq_avg_depth']


In [53]:
# Cell 7: Select required columns and save map dataset
map_cols = [
    'REPORT_NUM', 'IYEAR', 'SIGNIFICAN', 'CAUSE',
    'LOCATION_L', 'LOCATION_1', 'ONSHORE_ST',
    'TOTAL_COST', 'PIPE_DIAME', 'MATERIAL_I',
    'INSTALLATI', 'SYSTEM_PAR', 'HubDist',
    'eq_count', 'eq_max_mag', 'eq_avg_depth'
]

# Add pipeline type
hl_final['pipe_type'] = 'hazardous_liquid'
gas_final['pipe_type'] = 'gas'

# Combine both
df_map = pd.concat([
    hl_final[map_cols + ['pipe_type']],
    gas_final[map_cols + ['pipe_type']]
], ignore_index=True)

# Rename lat/lon columns
df_map = df_map.rename(columns={
    'LOCATION_L': 'latitude',
    'LOCATION_1': 'longitude'
})

# Save map dataset with original names
df_map.to_csv(r'D:\sfpris\data\processed\final_dataset_map.csv', index=False)

print(f"Map dataset saved: {df_map.shape}")
print(df_map.head())

Map dataset saved: (7958, 17)
   REPORT_NUM  IYEAR SIGNIFICAN                 CAUSE  latitude  longitude  \
0    20100001   2010         NO     EQUIPMENT FAILURE  41.94352  -88.23353   
1    20100002   2010         NO  OTHER ACCIDENT CAUSE  37.10847 -100.80037   
2    20100003   2010        YES     CORROSION FAILURE  32.22471 -101.40440   
3    20100004   2010         NO     EQUIPMENT FAILURE  40.60860  -74.23990   
4    20100005   2010         NO     CORROSION FAILURE  31.13284 -101.18974   

  ONSHORE_ST  TOTAL_COST  PIPE_DIAME                        MATERIAL_I  \
0         IL       30515         NaN  MATERIAL OTHER THAN CARBON STEEL   
1         KS        3510         NaN                      CARBON STEEL   
2         TX       26860       12.75                      CARBON STEEL   
3         NJ       17100         NaN  MATERIAL OTHER THAN CARBON STEEL   
4         TX       17485       10.00                      CARBON STEEL   

  INSTALLATI                                       SYSTE

In [54]:
# Cell 8: Prepare ML dataset
ml_cols = [
    'IYEAR', 'SIGNIFICAN', 'CAUSE', 'ONSHORE_ST',
    'TOTAL_COST', 'PIPE_DIAME', 'MATERIAL_I',
    'INSTALLATI', 'SYSTEM_PAR', 'HubDist',
    'eq_count', 'eq_max_mag', 'eq_avg_depth', 'pipe_type'
]

df = df_map[ml_cols].copy()

# Convert numeric columns
df['PIPE_DIAME'] = pd.to_numeric(df['PIPE_DIAME'], errors='coerce')
df['INSTALLATI'] = pd.to_numeric(df['INSTALLATI'], errors='coerce')

# Fill missing values
df['PIPE_DIAME'] = df['PIPE_DIAME'].fillna(df['PIPE_DIAME'].median())
df['INSTALLATI'] = df['INSTALLATI'].fillna(df['INSTALLATI'].median())
df['ONSHORE_ST'] = df['ONSHORE_ST'].fillna('UNKNOWN')

# Encode categorical columns
cat_cols = ['SIGNIFICAN', 'CAUSE', 'ONSHORE_ST', 
            'MATERIAL_I', 'SYSTEM_PAR', 'pipe_type']

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print(f"ML dataset shape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nDataset head:")
print(df.head())

ML dataset shape: (7958, 14)

Missing values:
IYEAR           0
SIGNIFICAN      0
CAUSE           0
ONSHORE_ST      0
TOTAL_COST      0
PIPE_DIAME      0
MATERIAL_I      0
INSTALLATI      0
SYSTEM_PAR      0
HubDist         0
eq_count        0
eq_max_mag      0
eq_avg_depth    0
pipe_type       0
dtype: int64

Dataset head:
   IYEAR  SIGNIFICAN  CAUSE  ONSHORE_ST  TOTAL_COST  PIPE_DIAME  MATERIAL_I  \
0   2010           0      1          12       30515       12.00           1   
1   2010           0      6          14        3510       12.00           0   
2   2010           1      0          41       26860       12.75           0   
3   2010           0      1          29       17100       12.00           1   
4   2010           0      0          41       17485       10.00           0   

   INSTALLATI  SYSTEM_PAR       HubDist  eq_count  eq_max_mag  eq_avg_depth  \
0      2000.0           8  1.189293e+06         3         3.2     14.783333   
1      1993.0           8  5.157653e+04  

In [55]:
# Cell 9: Split data and train XGBoost model
X = df.drop(columns=['SIGNIFICAN'])
y = df['SIGNIFICAN']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train XGBoost model
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)

model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)

print("Model trained successfully!")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))
print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model trained successfully!

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92       859
           1       0.96      0.83      0.89       733

    accuracy                           0.90      1592
   macro avg       0.91      0.90      0.90      1592
weighted avg       0.91      0.90      0.90      1592


Confusion Matrix:
[[832  27]
 [126 607]]


In [56]:
# Cell 10: Save model and final dataset
# Save ML dataset
df.to_csv(r'D:\sfpris\data\processed\final_dataset.csv', index=False)
print("ML dataset saved!")

# Save map dataset
print("Map dataset already saved!")

# Save model
joblib.dump(model, r'D:\sfpris\models\xgboost_pipeline_risk.pkl')
print("Model saved!")

print("\nAll files saved successfully!")
print(f"ML dataset: D:\\sfpris\\data\\processed\\final_dataset.csv")
print(f"Map dataset: D:\\sfpris\\data\\processed\\final_dataset_map.csv")
print(f"Model: D:\\sfpris\\models\\xgboost_pipeline_risk.pkl")

ML dataset saved!
Map dataset already saved!
Model saved!

All files saved successfully!
ML dataset: D:\sfpris\data\processed\final_dataset.csv
Map dataset: D:\sfpris\data\processed\final_dataset_map.csv
Model: D:\sfpris\models\xgboost_pipeline_risk.pkl


In [57]:
import geopandas as gpd
import os

# Load pipeline shapefiles
raw_dir = r'D:\sfpris\data\raw'

hl_network = gpd.read_file(f"{raw_dir}/CrudeOil_Pipelines_US_202001.shp")
gas_network = gpd.read_file(f"{raw_dir}/NaturalGas_Pipelines_US_202001.shp")

# Convert to WGS84
hl_network = hl_network.to_crs(epsg=4326)
gas_network = gas_network.to_crs(epsg=4326)

# Save as GeoJSON
hl_network.to_file(r'D:\sfpris\data\processed\hl_network.geojson', driver='GeoJSON')
gas_network.to_file(r'D:\sfpris\data\processed\gas_network.geojson', driver='GeoJSON')

print(f"HL network: {hl_network.shape}")
print(f"Gas network: {gas_network.shape}")
print("GeoJSON files saved!")

HL network: (236, 4)
Gas network: (32961, 5)
GeoJSON files saved!
